# 火灾疏散项目 — Colab 训练脚本
## 使用方法
1. 菜单栏 → **修改** → **笔记本设置** → 硬件加速器选 **T4 GPU**
2. 按 `Ctrl+F9` 全部运行，或逐个单元格点播放按钮
3. 训练完最后一步会弹出下载链接，把模型下载到本地

In [ ]:
# ============================================================
# 第 1 步：挂载 Google Drive（可选，跳过也不影响训练）
# 如果弹窗授权失败直接点"取消"，模型会通过浏览器下载
# ============================================================
import os
USE_DRIVE = False
DRIVE_PATH = '/content/drive/MyDrive/huo-zai-checkpoints'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    USE_DRIVE = True
    print(f'Google Drive 已挂载，模型将保存到: {DRIVE_PATH}')
except Exception as e:
    print(f'Google Drive 未挂载（{e}）')
    print('训练完会直接弹出下载链接，不影响使用')

In [ ]:
# ============================================================
# 第 2 步：克隆你的代码 + 安装依赖
# ============================================================
!git clone https://github.com/confidentismylife/LonlyStydue.git
%cd LonlyStydue
!pip install -q torch numpy pandas matplotlib scipy scikit-learn tqdm
print('依赖安装完成')

In [ ]:
# ============================================================
# 第 3 步：下载 ETH/UCY 数据集（约 500MB，只需下载一次）
# ============================================================
!mkdir -p datasets && cd datasets && wget -q https://github.com/crowdbotp/OpenTraj/raw/master/datasets/ETH_UCY/ethucy.zip && unzip -q ethucy.zip && rm ethucy.zip
%cd /content/LonlyStydue

import os
txt_count = sum(1 for root, dirs, files in os.walk('datasets') for f in files if f.endswith('.txt'))
print(f'数据集下载完成，{txt_count} 个 .txt 文件')

In [ ]:
# ============================================================
# 第 4 步：检查 GPU
# ============================================================
import torch
print(f'PyTorch: {torch.__version__}  |  GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# 第 5 步：加载数据 + 预处理
# ============================================================
import sys
sys.path.insert(0, '.')

import numpy as np
from src.data_loader import load_eth_ucy_file, extract_trajectories, normalize_trajectories
import src.config as cfg

all_files = []
for root, dirs, files in os.walk('/content/LonlyStydue/datasets'):
    for f in files:
        if f.endswith('.txt'):
            all_files.append(os.path.join(root, f))

print(f'找到 {len(all_files)} 个数据文件')

all_obs, all_preds = [], []
for fpath in all_files:
    raw = load_eth_ucy_file(fpath)
    obs, preds = extract_trajectories(raw)
    if len(obs) > 0:
        all_obs.append(obs)
        all_preds.append(preds)

all_obs = np.concatenate(all_obs)
all_preds = np.concatenate(all_preds)
print(f'总轨迹数: {len(all_obs):,}')

# 归一化
obs_norm, preds_norm = normalize_trajectories(all_obs, all_preds)

# 8:2 划分
split = int(len(obs_norm) * 0.8)
train_obs, train_preds = obs_norm[:split], preds_norm[:split]
val_obs, val_preds = obs_norm[split:], preds_norm[split:]
print(f'训练集: {len(train_obs):,}  |  验证集: {len(val_obs):,}')

In [ ]:
# ============================================================
# 第 6 步：训练！
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.model import SimpleLSTM
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TrajDataset(Dataset):
    def __init__(self, obs, preds):
        self.obs = torch.from_numpy(obs).float()
        self.preds = torch.from_numpy(preds).float()
    def __len__(self): return len(self.obs)
    def __getitem__(self, idx): return self.obs[idx], self.preds[idx]

train_loader = DataLoader(TrajDataset(train_obs, train_preds), batch_size=128, shuffle=True)
val_loader   = DataLoader(TrajDataset(val_obs, val_preds),   batch_size=128, shuffle=False)

model = SimpleLSTM(hidden_dim=64, pred_len=cfg.PRED_LEN).to(device)
print(f'模型参数: {sum(p.numel() for p in model.parameters()):,}  |  设备: {device}')

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10)

best_ade = float('inf')
EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ---- 训练阶段 ----
    model.train()
    train_loss = 0.0
    for obs, preds in train_loader:
        obs = obs.to(device)
        preds = preds.to(device)
        optimizer.zero_grad()
        output = model(obs)
        loss = criterion(output, preds)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * obs.size(0)
    train_loss /= len(train_loader.dataset)

    # ---- 验证阶段 ----
    model.eval()
    ade_sum = 0.0
    fde_sum = 0.0
    count = 0
    with torch.no_grad():
        for obs, preds in val_loader:
            obs = obs.to(device)
            preds = preds.to(device)
            diff = model(obs) - preds  # (batch, 12, 2)
            ade_sum += torch.norm(diff, dim=2).mean(dim=1).sum().item()
            fde_sum += torch.norm(diff[:, -1, :], dim=1).sum().item()
            count += obs.size(0)

    val_ade = ade_sum / count
    val_fde = fde_sum / count
    scheduler.step(val_ade)

    # ---- 保存最佳模型 ----
    if val_ade < best_ade:
        best_ade = val_ade
        os.makedirs('checkpoints', exist_ok=True)
        torch.save(model.state_dict(), 'checkpoints/best_model.pth')

    if epoch <= 3 or epoch % 20 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss:.4f} | '
              f'ADE: {val_ade:.3f}m | FDE: {val_fde:.3f}m | {time.time()-t0:.1f}s')

print(f'\n训练完成! 最佳验证 ADE: {best_ade:.3f} 米')
print(f'(匀速基线 ADE=0.48m, 你超过了它吗?)')

In [ ]:
# ============================================================
# 第 7 步：保存模型到 Google Drive（如果挂载了的话）
# ============================================================
if USE_DRIVE:
    import shutil
    shutil.copy('checkpoints/best_model.pth', os.path.join(DRIVE_PATH, 'best_model.pth'))
    print(f'模型已保存到 Google Drive: {DRIVE_PATH}/best_model.pth')
else:
    print('跳过 Google Drive 保存（未挂载）')

In [ ]:
# ============================================================
# 第 8 步：浏览器直接下载模型文件
# ============================================================
from google.colab import files
files.download('checkpoints/best_model.pth')
print('下载完成后，把 best_model.pth 放到你本地的 Huo-zai/checkpoints/ 目录下')

In [ ]:
# ============================================================
# 第 9 步：可视化预测效果
# ============================================================
import matplotlib.pyplot as plt

model.eval()
ids = np.random.choice(len(val_obs), 3, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, idx in enumerate(ids):
    obs_i = torch.from_numpy(val_obs[idx]).float().unsqueeze(0).to(device)
    gt = val_preds[idx]
    with torch.no_grad():
        pred = model(obs_i).squeeze(0).cpu().numpy()
    obs_i = obs_i.squeeze(0).cpu().numpy()

    ax = axes[i]
    ax.plot(obs_i[:, 0], obs_i[:, 1], 'b-o', label='Observed')
    ax.plot(pred[:, 0], pred[:, 1], 'r--o', label='Predicted')
    ax.plot(gt[:, 0], gt[:, 1], 'g-o', alpha=0.5, label='GT')
    ax.set_title(f'Sample {i+1}')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('colab_results.png', dpi=100)
plt.show()